In [1]:
from google.colab import drive
drive.mount('/content/drive')

import glob
checkpoints = glob.glob("/content/drive/MyDrive/dqn_checkpoints/*.pt")
for c in sorted(checkpoints):
    print(c)

Mounted at /content/drive
/content/drive/MyDrive/dqn_checkpoints/best_model.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_100000.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_1000000.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_1100000.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_1200000.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_1300000.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_1400000.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_1500000.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_1600000.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_1700000.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_1800000.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_1900000.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_200000.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_2000000.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_300000.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_400000.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_500000.pt
/content/drive/MyDrive/dqn_checkpoints/dqn_600000.pt
/content/

In [2]:
!pip install gymnasium[atari] ale-py moviepy -q

In [3]:
import os
import random
import time

from collections import deque
from dataclasses import dataclass, field
from typing import Tuple, List

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

import gymnasium as gym
import ale_py
from gymnasium.wrappers import (
    AtariPreprocessing,
    FrameStackObservation,
)

ale_py.register_v5_envs()

@dataclass
class Config:
    env_id = "ALE/Breakout-v5"
    seed = 42

    # start_learning: 80,000 means for the first 80k steps,
    # do absolutely nothing. this is to fill the replay buffer with enough
    # transition data for the model to start learning so that it doesnt overfit

    # train freq means train only every 4 steps. each step adds just 1 transition to the buffer
    #

    # target_update_freq:
    # every 10k steps copy the online network weights into
    # the target network. between copies the target stays frozen. without this the
    # bellman target y = r + gamma * max Q_target(s') shifts every single update
    total_steps = 2_000_000
    start_learning = 80_000
    train_freq = 4
    target_update = 10_000

    batch_size = 32
    gamma = 0.99
    lr = 1e-4
    grad_clip = 10.0 # clipping grads to prevent large af gradient updates

    buffer_size = 100_000

    # eps is used to control exploration - exploitation
    # initially, as eps is higher, the agent will prefer exploration
    # as it anneals every 500k steps, it will move towards exploitation as
    # it will learn stuff
    eps_st = 1.0
    eps_end = 0.01
    eps_anneal = 1_000_000

    log_interval = 1_000
    save_interval = 100_000
    save_dir = "checkpoints"


class ReplayBuffer:
    # circular q(ueue) implementation
    def __init__(self, capacity, obs_shape, device):
        self.capacity = capacity
        self.device = device

        self.pos = 0
        self.size = 0

        self.obs = np.zeros((capacity, *obs_shape), dtype = np.uint8)
        self.next_obs = np.zeros((capacity, *obs_shape), dtype = np.uint8)

        self.actions = np.zeros((capacity, ), dtype = np.int64)
        self.rewards = np.zeros((capacity, ), dtype = np.float32)
        self.dones = np.zeros((capacity, ), dtype = np.float32)

    def push(self, obs, action, reward, next_obs, done):
        # store a single transition
        self.obs[self.pos] = obs
        self.next_obs[self.pos] = next_obs
        self.actions[self.pos] = action
        self.rewards[self.pos] = reward
        self.dones[self.pos] = done

        self.pos= (self.pos + 1) % self.capacity
        self.size = min(self.size+1, self.capacity)

    def sample(self, batch_size):
        # sample a random batch of transitions and return as tensors
        idxs = np.random.randint(0, self.size, size = batch_size)

        # normalize since stored as uint
        obs = torch.tensor(self.obs[idxs], dtype = torch.float32,device=self.device) / 255.0
        next_obs = torch.tensor(self.next_obs[idxs], dtype = torch.float32,device=self.device) / 255.0

        actions = torch.tensor(self.actions[idxs], dtype = torch.int64,device = self.device)
        rewards = torch.tensor(self.rewards[idxs], dtype = torch.float32, device = self.device)
        dones= torch.as_tensor(self.dones[idxs],dtype=torch.float32, device=self.device)

        return obs, actions, rewards, next_obs, dones

    def __len__(self):
        return self.size

'''
input: stack of 4 grayscale frames: dims = 84x84 (4, 84, 84)
output: q value for every action (n_actions, )

Conv(32, 8x8, stride 4), Conv(64, 4x4, stride 2)
Conv(64, 3x3, stride 1), FC(512) → FC(n_actions)
'''
class DQN(nn.Module):
    def __init__(self, n_actions):
        super().__init__()

        # out_size = floor((input_size - kernel_size) / stride) + 1
        self.features = nn.Sequential(
            nn.Conv2d(4, 32, 8, 4),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, 1),
            nn.ReLU(),
            nn.Flatten()
        )

        self.head = nn.Sequential(
            nn.Linear(3136, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.head(self.features(x))

def select_action(qnet, obs, epsilon, n_actions, device):
    if random.random() < epsilon:
        return random.randrange(n_actions)

    else:
        obs_t = torch.as_tensor(obs, dtype = torch.float32, device = device).unsqueeze(0)/255.0
        with torch.no_grad():
            q_vals = qnet(obs_t)

        return int(q_vals.argmax(dim = 1).item())

def linear_epsilon(step, cfg):
    # linearly decay epsilon value from eps_start to eps_end
    fraction = min(step / cfg.eps_anneal, 1.0)
    return cfg.eps_st + fraction * (cfg.eps_end - cfg.eps_st)

# emulate an target net (separate from online net)
# that updates only every 10k steps
def compute_loss(online_net, target_net, batch, gamma):
    obs, actions, rewards, next_obs, dones = batch

    # current q_vals from the action taken
    q_vals = online_net(obs).gather(1, actions.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        max_next_q = target_net(next_obs).max(dim=1).values
        targets = rewards + gamma * max_next_q * (1.0 - dones)

    # choosing huber loss over MSE
    loss = nn.functional.smooth_l1_loss(q_vals, targets)
    return loss

def make_env(env_id: str, seed: int, render: bool = False):
    render_mode = "human" if render else None

    no_skip_id = env_id.replace("-v5", "NoFrameskip-v4").replace("ALE/", "")
    env = gym.make(no_skip_id, render_mode=render_mode)
    env = AtariPreprocessing(env, grayscale_obs=True, scale_obs=False, frame_skip=4)
    env = FrameStackObservation(env, stack_size=4)
    env.action_space.seed(seed)
    return env

def train(cfg):
    os.makedirs(cfg.save_dir, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    random.seed(cfg.seed)
    np.random.seed(cfg.seed)
    torch.manual_seed(cfg.seed)

    env = make_env(cfg.env_id, cfg.seed)
    n_actions = env.action_space.n
    obs_shape = env.observation_space.shape  # (4, 84, 84)
    print(f"Env: {cfg.env_id}  |  Actions: {n_actions}  |  Obs: {obs_shape}")

    online_net = DQN(n_actions).to(device)
    target_net = DQN(n_actions).to(device)
    target_net.load_state_dict(online_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(online_net.parameters(), lr=cfg.lr)
    buffer = ReplayBuffer(cfg.buffer_size, obs_shape, device)

    # METRICS
    ep_rewards = []
    ep_lens = []
    losses = []

    obs, _ = env.reset(seed=cfg.seed)
    ep_reward, ep_len = 0.0, 0
    start_time = time.time()

    print("STARTING TRAINING !!!!!!")

    '''
    for reference
    total_steps = 10_000_000
    start_learning = 80_000
    train_freq = 4
    target_update = 10_000
    '''

    for step in range(1, cfg.total_steps+1):
        eps = linear_epsilon(step, cfg)
        action = select_action(online_net, obs, eps, n_actions, device)

        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        clipped_rew = float(np.clip(reward, -1.0, 1.0))

        buffer.push(obs, action, clipped_rew, next_obs, terminated)
        obs = next_obs
        ep_reward+=reward
        ep_len += 1

        if done:
            ep_rewards.append(ep_reward)
            ep_lens.append(ep_len)

            obs, _ = env.reset()
            ep_reward, ep_len = 0.0,0

        if step >= cfg.start_learning and step % cfg.train_freq == 0:
            batch = buffer.sample(cfg.batch_size)
            loss  = compute_loss(online_net, target_net, batch, cfg.gamma)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(online_net.parameters(), cfg.grad_clip)
            optimizer.step()

            losses.append(loss.item())

        if step % cfg.target_update == 0:
            target_net.load_state_dict(online_net.state_dict())

        if step % cfg.log_interval == 0:
            sps = step / (time.time() - start_time)
            n_ep = len(ep_rewards)
            recent = ep_rewards[-20:] if n_ep else [0]
            mean_r = np.mean(recent)
            mean_l = np.mean(losses[-100:]) if losses else float("nan")

            print(
                f"step={step:>8,}  eps={eps:.3f}  "
                f"ep={n_ep:>6,}  mean_reward(20)={mean_r:>7.1f}  "
                f"loss={mean_l:.4f}  buf={len(buffer):>7,}  sps={sps:.0f}"
            )

        if step % cfg.save_interval == 0:
            path = os.path.join(cfg.save_dir, f"dqn_{step}.pt")
            torch.save({
                "step":       step,
                "online":     online_net.state_dict(),
                "target":     target_net.state_dict(),
                "optimizer":  optimizer.state_dict(),
                "ep_rewards": ep_rewards,
            }, path)
            print(f"  → checkpoint saved: {path}")

    env.close()

    final_path = os.path.join(cfg.save_dir, "best_model.pt")
    torch.save({
        "step":       cfg.total_steps,
        "online":     online_net.state_dict(),
        "target":     target_net.state_dict(),
        "optimizer":  optimizer.state_dict(),
        "ep_rewards": ep_rewards,
        "config":     cfg,
    }, final_path)
    print(f"\nTraining complete. Final model saved at {final_path}")
    return online_net, ep_rewards

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:637: UserWarning: WARN: Overriding environment ALE/Adventure-v5 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:637: UserWarning: WARN: Overriding environment ALE/AirRaid-v5 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:637: UserWarning: WARN: Overriding environment ALE/Alien-v5 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:637: UserWarning: WARN: Overriding environment ALE/Amidar-v5 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:637: UserW

In [4]:
import os
import random
import glob
import argparse

import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
import ale_py
from gymnasium.wrappers import (
    AtariPreprocessing,
    FrameStackObservation,
    RecordVideo,
)

ale_py.register_v5_envs()

class DQN(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(4, 32, 8, 4),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, 1),
            nn.ReLU(),
            nn.Flatten()
        )
        self.head = nn.Sequential(
            nn.Linear(3136, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions),
        )

    def forward(self, x):
        return self.head(self.features(x))


def select_action(net, obs, epsilon, n_actions, device):
    if random.random() < epsilon:
        return random.randrange(n_actions)
    obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0) / 255.0
    with torch.no_grad():
        q_vals = net(obs_t)
    return int(q_vals.argmax(dim=1).item())


def make_env(env_id, seed, record=False, video_dir="videos"):
    no_skip_id = env_id.replace("-v5", "NoFrameskip-v4").replace("ALE/", "")
    env = gym.make(no_skip_id, render_mode="rgb_array")
    env = AtariPreprocessing(env, grayscale_obs=True, scale_obs=False, frame_skip=4, terminal_on_life_loss=False)
    env = FrameStackObservation(env, stack_size=4)
    if record:
        os.makedirs(video_dir, exist_ok=True)
        env = RecordVideo(
            env,
            video_folder=video_dir,
            episode_trigger=lambda ep: True,
            name_prefix="dqn_play",
        )
    env.action_space.seed(seed)
    return env

model_path = "/content/drive/MyDrive/dqn_checkpoints/best_model.pt"
env_id     = "ALE/Breakout-v5"
n_episodes = 3
epsilon    = 0.0
seed       = 42
video_dir  = "videos"

device = torch.device("cpu")
env = make_env(env_id, seed, record=True, video_dir=video_dir)
n_actions = env.action_space.n

net = DQN(n_actions).to(device)
checkpoint = torch.load(model_path, map_location=device, weights_only=False)
net.load_state_dict(checkpoint["online"])
net.eval()
print(f"loaded checkpoint: {model_path}")
print(f"trained for {checkpoint.get('step', '?'):,} steps")

all_rewards = []
for ep in range(1, n_episodes + 1):
    obs, _ = env.reset(seed=seed + ep)
    total_reward, done = 0.0, False
    while not done:
        action = select_action(net, obs, epsilon, n_actions, device)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        done = terminated or truncated
    all_rewards.append(total_reward)
    print(f"  episode {ep}: reward = {total_reward:.1f}")

env.close()
print(f"\nmean reward: {np.mean(all_rewards):.1f}")
print(f"videos saved to: {os.path.abspath(video_dir)}/")


from google.colab import files
for f in glob.glob(f"{video_dir}/*.mp4"):
    files.download(f)

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


loaded checkpoint: /content/drive/MyDrive/dqn_checkpoints/best_model.pt
trained for 2,000,000 steps
  episode 1: reward = 175.0


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


  episode 2: reward = 191.0
  episode 3: reward = 191.0

mean reward: 185.7
videos saved to: /content/videos/


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>